In [ ]:
import sqlite3

database_path = '../database/db-2025-03-26.sqlite'
project_id = 6
conn = sqlite3.connect(database_path)

In [ ]:
import pandas as pd

# Query to get mutation status counts by project and stage
query = """
SELECT 
    p.id AS project_id,
    p.root_path,
    pmr.stage,
    pmr.variant,
    pmr.status,
    COUNT(*) AS mutation_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, p.root_path, pmr.stage, pmr.variant, pmr.status
ORDER BY 
    p.id, pmr.stage, pmr.variant, pmr.status
"""

# Query to get detected mutation counts
detected_query = """
SELECT 
    p.id AS project_id,
    pmr.stage,
    pmr.variant,
    SUM(pmr.is_detected) AS detected_count,
    COUNT(*) AS total_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, pmr.stage, pmr.variant
"""

# Execute queries and load results into DataFrames
mutation_stats = pd.read_sql_query(query, conn)
detected_stats = pd.read_sql_query(detected_query, conn)

# Function to create display variant names
def get_display_variant(row):
    if row['stage'] == 'COLLECT_PIT_DATA_ORIGINAL' and pd.isna(row['variant']):
        return 'ORIGINAL'
    elif row['stage'] == 'COLLECT_PIT_DATA_INITIAL' and pd.isna(row['variant']):
        return 'INITIAL'
    else:
        return row['variant']

# Create display variant columns
mutation_stats['display_variant'] = mutation_stats.apply(get_display_variant, axis=1)
detected_stats['display_variant'] = detected_stats.apply(get_display_variant, axis=1)

# Calculate detection percentage
detected_stats['DETECTED_PERCENTAGE'] = (detected_stats['detected_count'] / detected_stats['total_count'] * 100).round(2)

# Create a pivot table for mutation status counts
pivot_table = mutation_stats.pivot_table(
    index=['project_id', 'root_path', 'stage', 'display_variant'],
    columns='status',
    values='mutation_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Calculate total mutations
pivot_table['TOTAL'] = pivot_table.select_dtypes(include=['int64']).sum(axis=1)

# Merge the detected percentage into the pivot table
pivot_table = pd.merge(
    pivot_table,
    detected_stats[['project_id', 'stage', 'display_variant', 'DETECTED_PERCENTAGE']],
    on=['project_id', 'stage', 'display_variant'],
    how='left'
)

# Define sort orders
stage_order = {'COLLECT_PIT_DATA_ORIGINAL': 0, 'COLLECT_PIT_DATA_INITIAL': 1, 'COLLECT_PIT_DATA_GENERALIZED': 2}
variant_order = {'ORIGINAL': 0, 'INITIAL': 1, 'BASELINE': 2, 'IMPROVED_5_TRIES': 3, 'IMPROVED_20_TRIES': 4}

# Add sorting columns
pivot_table['stage_order'] = pivot_table['stage'].map(stage_order)
pivot_table['variant_order'] = pivot_table['display_variant'].map(variant_order)

# Sort the DataFrame
pivot_table = pivot_table.sort_values(['project_id', 'stage_order', 'variant_order'])

# Create a dictionary to store INITIAL detection percentages for each project
initial_detection = {
    row['project_id']: row['DETECTED_PERCENTAGE'] 
    for _, row in pivot_table[pivot_table['display_variant'] == 'INITIAL'].iterrows()
}

# Add improvement column
def calculate_improvement(row):
    if row['display_variant'] in ['ORIGINAL', 'INITIAL']:
        return None  # No improvement value for ORIGINAL and INITIAL
    initial = initial_detection.get(row['project_id'])
    if initial is None:
        return None  # No INITIAL variant to compare with
    return round(row['DETECTED_PERCENTAGE'] - initial, 2)

# Apply the function to create the new column
pivot_table['IMPROVEMENT'] = pivot_table.apply(calculate_improvement, axis=1)

# Extract project name from root_path for cleaner display
pivot_table['project_name'] = pivot_table['root_path'].apply(lambda x: x.split('/')[-1])

# Clean up the DataFrame and select/reorder columns
result_columns = ['project_id', 'project_name', 'stage', 'display_variant'] + \
                [col for col in pivot_table.columns if col not in 
                 ['project_id', 'project_name', 'root_path', 'stage', 'display_variant', 'stage_order', 'variant_order']]

result_table = pivot_table[result_columns].rename(columns={'display_variant': 'variant'})

# Display the results
print("Mutation Status Counts by Project, Stage, and Variant with Improvement Analysis")
display(result_table)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import OrderedDict
from matplotlib.gridspec import GridSpec

# Query to get mutation results by mutator type
mutator_query = """
SELECT 
    p.id AS project_id,
    p.root_path,
    pmr.stage,
    pmr.variant,
    pmr.mutator,
    pmr.is_detected,
    COUNT(*) AS mutation_count
FROM 
    pit_mutation_report pmr
JOIN 
    project p ON pmr.project_id = p.id
GROUP BY 
    p.id, p.root_path, pmr.stage, pmr.variant, pmr.mutator, pmr.is_detected
ORDER BY 
    p.id, pmr.stage, pmr.variant, pmr.mutator
"""

# Execute query and load results into DataFrame
mutator_stats = pd.read_sql_query(mutator_query, conn)

# Function to create display variant names
def get_display_variant(row):
    if row['stage'] == 'COLLECT_PIT_DATA_ORIGINAL' and pd.isna(row['variant']):
        return 'ORIGINAL'
    elif row['stage'] == 'COLLECT_PIT_DATA_INITIAL' and pd.isna(row['variant']):
        return 'INITIAL'
    else:
        return row['variant']

# Create display variant column
mutator_stats['display_variant'] = mutator_stats.apply(get_display_variant, axis=1)

# Extract project name from root_path
mutator_stats['project_name'] = mutator_stats['root_path'].apply(lambda x: x.split('/')[-1])

# Function to simplify mutator names (extract class name only)
def simplify_mutator_name(mutator):
    # Extract the class name from the fully qualified name
    match = re.search(r'\.([^.]+)$', mutator)
    if match:
        return match.group(1)
    return mutator

# Apply the function to create a simplified mutator column
mutator_stats['simple_mutator'] = mutator_stats['mutator'].apply(simplify_mutator_name)

# Create a pivot table for mutator analysis
mutator_pivot = mutator_stats.pivot_table(
    index=['project_id', 'project_name', 'stage', 'display_variant', 'simple_mutator'],
    columns='is_detected',
    values='mutation_count',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Rename columns for clarity
mutator_pivot = mutator_pivot.rename(columns={0: 'not_detected', 1: 'detected'})

# Calculate total mutations and detection rate for each mutator
mutator_pivot['total'] = mutator_pivot['not_detected'] + mutator_pivot['detected']
mutator_pivot['detection_rate'] = (mutator_pivot['detected'] / mutator_pivot['total'] * 100).round(2)

# Define sort orders
stage_order = {'COLLECT_PIT_DATA_ORIGINAL': 0, 'COLLECT_PIT_DATA_INITIAL': 1, 'COLLECT_PIT_DATA_GENERALIZED': 2}
variant_order = {'ORIGINAL': 0, 'INITIAL': 1, 'BASELINE': 2, 'IMPROVED_5_TRIES': 3, 'IMPROVED_20_TRIES': 4}

# Add sorting columns
mutator_pivot['stage_order'] = mutator_pivot['stage'].map(stage_order)
mutator_pivot['variant_order'] = mutator_pivot['display_variant'].map(lambda x: variant_order.get(x, 999))

# Sort the DataFrame
mutator_pivot = mutator_pivot.sort_values(['project_id', 'stage_order', 'variant_order', 'simple_mutator'])

# Create a dictionary to store INITIAL detection rates for each project and mutator
initial_detection_by_mutator = {}
for _, row in mutator_pivot[mutator_pivot['display_variant'] == 'INITIAL'].iterrows():
    key = (row['project_id'], row['simple_mutator'])
    initial_detection_by_mutator[key] = row['detection_rate']

# Add improvement column
def calculate_mutator_improvement(row):
    if row['display_variant'] in ['ORIGINAL', 'INITIAL']:
        return None  # No improvement value for ORIGINAL and INITIAL
    key = (row['project_id'], row['simple_mutator'])
    initial = initial_detection_by_mutator.get(key)
    if initial is None:
        return None  # No INITIAL variant to compare with
    return round(row['detection_rate'] - initial, 2)

# Apply the function to create the new column
mutator_pivot['improvement'] = mutator_pivot.apply(calculate_mutator_improvement, axis=1)

# Clean up the DataFrame
result_columns = ['project_id', 'project_name', 'stage', 'display_variant', 'simple_mutator', 
                 'detected', 'not_detected', 'total', 'detection_rate', 'improvement']
mutator_results = mutator_pivot[result_columns].rename(columns={'display_variant': 'variant', 'simple_mutator': 'mutator'})

# Define consistent colors for variants using a colorblind-friendly palette
variant_colors = {
    'ORIGINAL': '#1f77b4',  # blue
    'INITIAL': '#ff7f0e',   # orange
    'BASELINE': '#2ca02c',  # green
    'IMPROVED_5_TRIES': '#d62728',  # red
    'IMPROVED_20_TRIES': '#9467bd'  # purple
}

# Get unique variants for analysis
variants_to_include = ['ORIGINAL', 'INITIAL', 'BASELINE', 'IMPROVED_5_TRIES', 'IMPROVED_20_TRIES']
variants_to_include = [v for v in variants_to_include if v in mutator_results['variant'].unique()]

# Calculate average improvement by mutator type
improvement_by_mutator = mutator_results[mutator_results['improvement'].notna()].groupby('mutator')['improvement'].mean().sort_values(ascending=False)

# Create a bar chart of average improvement by mutator
plt.figure(figsize=(12, 6))
bars = improvement_by_mutator.plot(kind='bar', color='#2ca02c')  # Use green for improvement bars
plt.title('Average Improvement in Detection Rate by Mutator Type')
plt.xlabel('Mutator')
plt.ylabel('Average Improvement (%)')
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Get unique projects
projects = mutator_results['project_name'].unique()
project_count = len(projects)

# Get all unique mutators across all projects to ensure consistent x-axis
all_mutators = sorted(mutator_results['mutator'].unique())
mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}

# Define improvement variants once
improvement_variants = [v for v in variants_to_include if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]

# Create a figure with custom grid layout (detection rate plots wider than improvement plots)
fig = plt.figure(figsize=(16, 4 * project_count))  # Reduced height per project
gs = GridSpec(project_count, 2, width_ratios=[3, 2])

# Set fixed x-axis limits for all plots
x_min = -0.5
x_max = len(all_mutators) - 0.5

# Set up consistent y-axis limits
detection_y_max = 0
improvement_y_max = 0
improvement_y_min = 0

# First pass to determine y-axis limits
for project in projects:
    project_data = mutator_results[mutator_results['project_name'] == project]
    detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

    project_improvements = project_data[project_data['improvement'].notna()]['improvement']
    if not project_improvements.empty:
        improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
        improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

# Create plots for each project
for i, project in enumerate(projects):
    # Filter data for this project
    project_data = mutator_results[mutator_results['project_name'] == project]

    # Get the axes for this project
    ax1 = fig.add_subplot(gs[i, 0])
    ax2 = fig.add_subplot(gs[i, 1])

    # 1. Detection Rate Chart
    # Calculate the center position for each group of bars
    width = 0.15
    num_variants = len(variants_to_include)
    total_width = width * num_variants
    group_offsets = -total_width/2 + width/2  # Start offset to center the group

    # Plot each variant as a group of bars
    for variant_idx, variant in enumerate(variants_to_include):
        variant_data = project_data[project_data['variant'] == variant]

        # Create a dictionary mapping mutator to detection rate for this variant
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        # For each mutator position, plot the bar if data exists
        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            offset = group_offsets + width * variant_idx
            ax1.bar(pos + offset, rate, width, label=variant if pos == 0 else "", color=variant_colors[variant])

    # Add labels and title
    ax1.set_ylabel('Detection Rate (%)')
    ax1.set_title(f'Detection Rate - {project}')

    # Set fixed axis limits
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(0, detection_y_max)

    # Only add x-tick labels for the last project
    if i == project_count - 1:
        ax1.set_xticks(np.arange(len(all_mutators)))
        ax1.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax1.set_xticks(np.arange(len(all_mutators)))
        ax1.set_xticklabels([])

    # Move legend outside the plot to avoid overlap
    if i == 0:
        ax1.legend(loc='upper center', bbox_to_anchor=(0.5, 1.45), ncol=len(variants_to_include))

    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # 2. Improvement Chart
    width = 0.22
    num_improvement_variants = len(improvement_variants)

    if num_improvement_variants > 0:
        # Calculate the center position for each group of bars
        total_width = width * num_improvement_variants
        group_offsets = -total_width/2 + width/2  # Start offset to center the group

        # Plot each variant as a group of bars (excluding ORIGINAL and INITIAL)
        for variant_idx, variant in enumerate(improvement_variants):
            variant_data = project_data[project_data['variant'] == variant]

            # Create a dictionary mapping mutator to improvement for this variant
            improvements = {row['mutator']: row['improvement'] for _, row in variant_data.iterrows() if row['improvement'] is not None}

            # For each mutator position, plot the bar if data exists
            for mutator, pos in mutator_positions.items():
                impr = improvements.get(mutator, 0)
                offset = group_offsets + width * variant_idx
                ax2.bar(pos + offset, impr, width, label=variant if pos == 0 else "", color=variant_colors[variant])

    # Add labels and title
    ax2.set_ylabel('Improvement (%)')
    ax2.set_title(f'Improvement - {project}')

    # Set fixed axis limits
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(improvement_y_min, improvement_y_max)

    # Only add x-tick labels for the last project
    if i == project_count - 1:
        ax2.set_xticks(np.arange(len(all_mutators)))
        ax2.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax2.set_xticks(np.arange(len(all_mutators)))
        ax2.set_xticklabels([])

    # Move legend outside the plot to avoid overlap
    if i == 0 and num_improvement_variants > 0:
        ax2.legend(loc='upper center', bbox_to_anchor=(0.5, 1.45), ncol=len(improvement_variants))

    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    ax2.axhline(y=0, color='r', linestyle='-', alpha=0.3)

# Adjust layout with more space at the top for legends
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.subplots_adjust(hspace=0.3, top=0.9)
plt.show()
